## Interactive audio sample cleaning

TODO:

* The `SpectrogramAnnotator` class silently screws up time scale when not using 32 kHz
* Add a primary_label column to metadata

In [1]:
from pathlib import Path
project_root = str(Path().resolve().parent)

In [2]:
geographic_extents = {'new_zealand': {'min_longitude': 166,
                                        'max_longitude': 179,
                                        'min_latitude': -49,
                                        'max_latitude': -34},
                      'south_america': {'min_longitude': -82,
                                        'max_longitude': -34.0,
                                        'min_latitude': -56.0,
                                        'max_latitude': 13.0}
                      }

### This cell requires manual setup

In [3]:
use_case = {
            'project_root': project_root,
            'source_dataset_path': project_root + '/data/anqa_from_avianz',
            'reviewed_data_destn':project_root + '/data/anqa_reviewed',
            'naming_csv':  project_root + '/data/bird_names/bird_names.csv',  #Requires eBird and CommonName columns, and must include Unknown | unknown
            'audio_folder':  Path(project_root + '/data/anqa_from_avianz/audio'),
            'author': 'Olly',   #Use Author for first annotation, otherwise None
            'reviewer': None,   #Use Reviwer for subsequent corrections/additions
            'map_extents': geographic_extents['new_zealand'],
            'display_width': 18, #Adjust for screen size, 
            }

load_samples = False

In [4]:
class FilePaths:
    def __init__(self, options: dict):
        _project_dir = Path(options['project_root'])
        self.original_dataset = Path(options['source_dataset_path'])
        self.data_folder = _project_dir / 'data'
        self.audio_folder = options.get('audio_folder', self.data_folder)
        self.original_labels = self.original_dataset / 'annotations.parquet'
        self.original_metadata = self.original_dataset / 'metadata.parquet'
        self.out_dataset = Path(options['reviewed_data_destn'])
        self.out_dataset.mkdir(exist_ok=True, parents=True)
        self.out_labels = self.out_dataset / 'annotations.parquet'
        self.out_metadata = self.out_dataset / 'metadata.parquet'
        self.naming_csv = Path(options['naming_csv'])

In [5]:
from __future__ import annotations  #Not needed for Python 3.10+
from datetime import date
import ast
import matplotlib
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import ipywidgets as widgets
button = widgets.Button(description="Continue")
output = widgets.Output()
import librosa
import time
import matplotlib.gridspec as gridspec
import threading
import contextily as cx
from scipy.signal import butter, filtfilt, resample_poly, find_peaks
from math import gcd
from threading import Timer
from dataclasses import dataclass
import sounddevice as sd

%matplotlib widget  
print(f'The Matplotlib backend is {matplotlib.get_backend()}')

The Matplotlib backend is widget


#### Helper Functions and Classes

In [6]:
from anqa.annotation import (MiniBirdNamer, FastMap, AnnotationState,
                             normalize_secondary_labels, create_class_widgets,
                             SpectrogramAnnotator, AnnotationSession,
                             AnnotationControls, load_current_sample)

### Initialize

In [7]:
paths = FilePaths(use_case)
namer = MiniBirdNamer(paths.naming_csv)
map = FastMap(map_extents=use_case['map_extents'],
              provider = cx.providers.Esri.WorldImagery) #.OpenTopoMap, OpenStreetMap.Mapnik, 
annotation_state = AnnotationState(all_classes=namer.common_names, max_visible=30)

## Check the data tables
Start by loading the metadata dataframe, there should be exactly one row per file

In [8]:
df_meta = pd.read_parquet(paths.original_metadata)
df_meta = df_meta.sort_values(by='filename')  #Ensures all the files from one class folder are presented sequentially
df_meta['secondary_labels'] = df_meta['secondary_labels'].apply(normalize_secondary_labels)
df_meta.head()

,filename,collection,secondary_labels,url,latitude,longitude,author,license,recorded_on,reviewed_by,reviewed_on,source_filename,source_start_s,source_end_s,models_used
30,20190831_074504_from_0.flac,NaN,[],NaN,NaN,NaN,Sumudu,NaN,2019-08-31 07:45:04,NaN,NaN,20190831_074504.wav,0.0,60.0,NaN
32,20190831_074504_from_120.flac,NaN,[],NaN,NaN,NaN,Sumudu,NaN,2019-08-31 07:45:04,NaN,NaN,20190831_074504.wav,120.0,180.0,NaN
33,20190831_074504_from_180.flac,NaN,[],NaN,NaN,NaN,Sumudu,NaN,2019-08-31 07:45:04,NaN,NaN,20190831_074504.wav,180.0,240.0,NaN
34,20190831_074504_from_240.flac,NaN,[],NaN,NaN,NaN,Sumudu,NaN,2019-08-31 07:45:04,NaN,NaN,20190831_074504.wav,240.0,300.0,NaN
35,20190831_074504_from_300.flac,NaN,[],NaN,NaN,NaN,Sumudu,NaN,2019-08-31 07:45:04,NaN,NaN,20190831_074504.wav,300.0,360.0,NaN


In [9]:
len(df_meta)

45

In [10]:
df_labels = pd.read_parquet(paths.original_labels)
df_labels.head(3)

,Filename,Start Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),Label,Type,Sex,Score,Delta Time (s),Delta Freq (Hz),Avg Power Density (dB FS/Hz)
0,20190831_083004_from_420.flac,37.6,45.0,1230.0,2365.0,weka1,<NA>,<NA>,NaN,7.4,1135.0,-53.0
1,20190831_181504_from_240.flac,54.5,60.0,997.0,2656.0,weka1,<NA>,<NA>,NaN,5.5,1659.0,-62.1
2,20190831_181504_from_300.flac,0.0,5.1,997.0,2656.0,weka1,<NA>,<NA>,NaN,5.1,1659.0,-61.2


In [11]:
df_naming = pd.read_csv(paths.naming_csv)
df_naming.head(3)

,CommonName,eBird,ScientificName,ExtraName
0,Bellbird,nezbel1,Anthornis melanura,Bellbird
1,Australasian Bittern,ausbit1,Botaurus poiciloptilus,Bittern
2,Bittern,nezbit1,Ixobrychus novaezelandiae,Bittern


## Annotation
* **Every** Bird or Animal Sound to be boxed
* Unknown classes to be labelled *Unknown*
* Calls from the same bird with gaps of greater than 2 seconds should have individual boxes
* Otherwise a single large box should be used
* Spacebar to clear all labels
* Right click to clear the last label
* The t key will attempt to box all patterns matching the contents of the most recent box

In [13]:
create_class_widgets(annotation_state, n_columns=6, fastmap=map, common_to_ebird=namer.common_to_ebird_dict)
annotator = SpectrogramAnnotator(annotation_state,
                                 #voice_detector=None, #voice_detect,
                                 common_to_ebird=namer.common_to_ebird_dict,
                                 plot_size = (use_case['display_width'],4),  #Adjust for screen size
                                 f_min=20,
                                 f_max=16000,
                                 zoom_window_height=0.4,
                                 zoom_window_width=5,
                                 min_drag_rows=5,
                                 min_drag_time_s=.1,
                                 min_separation = 2,
                                 similarness_threshold=0.5,
                                 min_freq_hz=300)    ####################Not working yet #########################

session = AnnotationSession(df_meta=df_meta,
                            df_labels=df_labels,
                            new_meta_filepath=paths.out_metadata,
                            new_labels_filepath=paths.out_labels,
                            reviewer = use_case['reviewer'],
                            author = use_case['author'])
controls = AnnotationControls()
annotation_widget = load_current_sample(session, annotator, paths, map)
controls.display()
controls.bind(session, annotator, paths, map)

def on_space_key(event):
    if event.key == ' ':
        controls._on_next_clicked()

annotator.fig.canvas.mpl_connect('key_press_event', on_space_key)

20

### Progress Checks

In [ ]:
session.summary()

{'total_files': 45,
 'finished_files_in_new_meta': 0,
 'total_annotations': 44,
 'done_in_current_session': np.int64(0),
 'pending_in_current_session': np.int64(45),
 'total_minus_pending': np.int64(0)}

In [ ]:
marked_times = annotator.get_boxes()
marked_times[-2:]

[{'Filename': '20190831_074504_from_0.flac',
  'Start Time (s)': 4.3,
  'End Time (s)': 4.6,
  'Low Freq (Hz)': 9728.0,
  'High Freq (Hz)': 11184.0,
  'Delta Time (s)': 0.3,
  'Delta Freq (Hz)': 1456.0,
  'Avg Power Density (dB FS/Hz)': -60.4,
  'Label': 'riflem1',
  'Type': <NA>,
  'Sex': <NA>,
  'Score': nan,
  'Life Stage': None},
 {'Filename': '20190831_074504_from_0.flac',
  'Start Time (s)': 51.9,
  'End Time (s)': 52.1,
  'Low Freq (Hz)': 9259.0,
  'High Freq (Hz)': 9716.0,
  'Delta Time (s)': 0.2,
  'Delta Freq (Hz)': 457.0,
  'Avg Power Density (dB FS/Hz)': -55.6,
  'Label': 'riflem1',
  'Type': <NA>,
  'Sex': <NA>,
  'Score': nan,
  'Life Stage': None}]

In [ ]:
if paths.out_metadata.exists():
    output_metadata = pd.read_parquet(paths.out_metadata)
    display(output_metadata.tail())

In [ ]:
if paths.out_metadata.exists():
    display(output_metadata.shape)

In [ ]:
if paths.out_labels.exists():
    output_labeldata = pd.read_parquet(paths.out_labels)
    display(output_labeldata.tail(5))